# Sigchos Agrotech — Modelo TFLite v2 (con rechazo `no_hoja`)

Clasificador de hojas de **zapallo** (MobileNetV2 + transfer learning). A diferencia de la v1, agrega una 6ª clase **`no_hoja`** para rechazar fotos que NO son hoja de zapallo (objetos, fondos, hojas de otras plantas).

**Antes de correr:** Runtime → Change runtime type → **GPU (T4)**.

**Necesitas:** token de Kaggle (https://www.kaggle.com/settings → API → Create New Token).

**Recomendado (mejora mucho el rechazo):** sube ~150–300 fotos TUYAS que NO sean hoja (manos, tierra, piso, objetos, pared) a una carpeta `/content/mis_negativos` usando el icono de carpeta de Colab, antes de correr la celda 5.

## 1. Instalar dependencias

In [ ]:
!pip install -q kagglehub tensorflow scikit-learn

## 2. Autenticar y descargar datasets

Baja el dataset de zapallo (5 clases) y el dataset New Plant Diseases (se usa como fuente de "hojas de otras plantas" para la clase `no_hoja`).

In [ ]:
import kagglehub, os

kagglehub.login()
pumpkin_path = kagglehub.dataset_download("tahmidmir/pumpkin-leaf-diseases-dataset-from-bangladesh")
neg_path = kagglehub.dataset_download("vipoooool/new-plant-diseases-dataset")
print("Zapallo en:", pumpkin_path)
print("Negativos (otras plantas) en:", neg_path)

## 3. Mapear carpetas del dataset de zapallo → clases de la app

Ajusta las CLAVES (izquierda) si no calzan con los nombres reales de carpeta. Los VALORES (derecha) son los `claseId` de la app — no los cambies.

In [ ]:
MAPEO_CARPETAS = {
    "Healthy Leaf": "hoja_sana",
    "Downy Mildew": "mildiu",
    "Powdery Mildew": "oidio",
    "Mosaic Disease": "amarillamiento",
    "Bacterial Leaf Spot": "mancha_foliar",
}

## 4. Reorganizar zapallo + construir la clase `no_hoja`

`no_hoja` = hojas de OTRAS plantas (enseña "es hoja pero no zapallo") + tus fotos no-hoja de `/content/mis_negativos` (enseña "ni siquiera es hoja").

In [ ]:
import shutil, glob, random

DATASET_LISTO = "/content/dataset_listo"
shutil.rmtree(DATASET_LISTO, ignore_errors=True)
os.makedirs(DATASET_LISTO, exist_ok=True)

EXTS = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')

def _norm(s):
    return s.strip().lower().replace('_', ' ').replace('-', ' ')

# --- Zapallo: buscar carpetas con imágenes ---
disponibles = {}
for root, dirs, files in os.walk(pumpkin_path):
    imgs = [f for f in files if f.endswith(EXTS)]
    if imgs:
        disponibles[_norm(os.path.basename(root))] = (root, len(imgs))

for carpeta, clase in MAPEO_CARPETAS.items():
    clave = _norm(carpeta)
    if clave not in disponibles:
        raise ValueError(f"No se encontró '{carpeta}'. Ajusta MAPEO_CARPETAS. Disponibles: {list(disponibles)}")
    shutil.copytree(disponibles[clave][0], os.path.join(DATASET_LISTO, clase), dirs_exist_ok=True)
    print(clase, '->', len(os.listdir(os.path.join(DATASET_LISTO, clase))), 'imgs')

# --- Cuántas imágenes poner en no_hoja (promedio de las clases de zapallo) ---
clases_zapallo = list(set(MAPEO_CARPETAS.values()))
conteos = [len(os.listdir(os.path.join(DATASET_LISTO, c))) for c in clases_zapallo]
objetivo = int(sum(conteos) / len(conteos))
print('Objetivo de imágenes para no_hoja:', objetivo)

# --- Construir no_hoja ---
no_dir = os.path.join(DATASET_LISTO, "no_hoja")
os.makedirs(no_dir, exist_ok=True)

# 1) Hojas de otras plantas
otras = []
for e in ('.jpg', '.JPG', '.jpeg', '.png'):
    otras += glob.glob(os.path.join(neg_path, '**', f'*{e}'), recursive=True)
random.seed(42)
random.shuffle(otras)
copiadas = 0
for src in otras:
    if copiadas >= objetivo:
        break
    try:
        shutil.copy(src, os.path.join(no_dir, f"otra_{copiadas}.jpg"))
        copiadas += 1
    except Exception:
        pass

# 2) Tus fotos no-hoja (opcional pero MUY recomendado)
mis = glob.glob("/content/mis_negativos/*")
for i, src in enumerate(mis):
    try:
        shutil.copy(src, os.path.join(no_dir, f"mio_{i}{os.path.splitext(src)[1]}"))
    except Exception:
        pass

print('no_hoja ->', len(os.listdir(no_dir)), 'imgs  (otras plantas:', copiadas, '+ tuyas:', len(mis), ')')
if len(mis) == 0:
    print('\u26a0\ufe0f  No subiste fotos propias a /content/mis_negativos. El rechazo de objetos no-hoja será más débil.')

## 5. Cargar datasets de entrenamiento / validación

In [ ]:
import tensorflow as tf
import numpy as np

IMG_SIZE = 224
BATCH_SIZE = 32
SEED = 42

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_LISTO, validation_split=0.2, subset="training", seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_LISTO, validation_split=0.2, subset="validation", seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
)

CLASES = train_ds.class_names  # orden alfabético = orden de salida del modelo
print("Clases (orden del modelo):", CLASES)

# class_weight balanceado (no_hoja puede desbalancear)
from sklearn.utils.class_weight import compute_class_weight
y_train = np.concatenate([y.numpy() for _, y in train_ds])
pesos = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight = dict(enumerate(pesos))
print("class_weight:", class_weight)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

## 6. Construir el modelo (MobileNetV2 + transfer learning)

Usa `preprocess_input` de MobileNetV2 — misma normalización `(pixel-127.5)/127.5` que la app. No hay que tocar código Flutter.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomContrast(0.1),
])

preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet"
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = data_augmentation(inputs)
x = preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(CLASES), activation="softmax")(x)
model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()

## 7. Entrenar (fase 1: solo la cabeza nueva)

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=12, class_weight=class_weight)

## 8. Fine-tuning (fase 2: descongelar últimas capas)

In [ ]:
base_model.trainable = True
for capa in base_model.layers[:-30]:
    capa.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

history_fine = model.fit(train_ds, validation_data=val_ds, epochs=8, class_weight=class_weight)

## 9. Evaluar (matriz de confusión — revisa recall de `hoja_sana` y `no_hoja`)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true, y_pred = [], []
for imagenes, etiquetas in val_ds:
    preds = model.predict(imagenes, verbose=0)
    y_true.extend(etiquetas.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=CLASES))
print("Matriz de confusión:\n", confusion_matrix(y_true, y_pred))

## 10. Convertir a TFLite (float16 — preciso y liviano)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

with open("model.tflite", "wb") as f:
    f.write(tflite_model)
with open("labels.txt", "w") as f:
    f.write("\n".join(CLASES))

print("model.tflite:", round(len(tflite_model) / 1024, 1), "KB")
print("labels.txt (orden del modelo):", CLASES)

## 11. Descargar

In [ ]:
from google.colab import files
files.download("model.tflite")
files.download("labels.txt")

**Último paso (en tu PC):** copia `model.tflite` y `labels.txt` a `mobile_app/assets/ml/` (reemplaza). Luego:
```
cd mobile_app
flutter clean
flutter pub get
flutter run
```
La app ya trata `no_hoja` como rechazo. No requiere cambios de código.